# 🎓 Day 1 – Session 1
# How Large Language Models Really Work
## From Next-Token Prediction to Intelligent AI Systems

Welcome to the FDP on **Agentic AI**.

### Learning Objectives
By the end of this notebook you will:
- Understand tokenization
- Estimate token cost
- Learn next-token prediction
- Explore temperature
- Understand context windows
- Demonstrate hallucinations
- See why RAG is needed


## 📦 Install Packages (Run only in a fresh environment)

In [ ]:
%pip install python-dotenv tiktoken pandas openai



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 📚 Import Libraries

In [1]:
import os
from pathlib import Path

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

## 🔑 Configure OpenAI API Key

In [3]:



# Go one level up from the notebook folder to the repository root
env_path = Path.cwd().parent / ".env"


print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)

load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file location and name.")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")

Current working directory: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai/day1-llm-foundations
Looking for .env at: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.env
OpenAI client initialized successfully.


# 1️⃣ Tokenization

LLMs don't read words—they read **tokens**.

Why it matters:
- Pricing
- Context window
- Performance


In [4]:
enc=tiktoken.get_encoding("cl100k_base")

text="Artificial Intelligence is transforming education."
tokens=enc.encode(text)

print(text)
print("\nToken IDs:",tokens)
print("\nToken Count:",len(tokens))

print("\nDecoded Tokens")
for t in tokens:
    print(t,"->",enc.decode([t]))


Artificial Intelligence is transforming education.

Token IDs: [9470, 16895, 22107, 374, 46890, 6873, 13]

Token Count: 7

Decoded Tokens
9470 -> Art
16895 -> ificial
22107 ->  Intelligence
374 ->  is
46890 ->  transforming
6873 ->  education
13 -> .


### 🧪 Exercise: Compare token counts for English and Indian languages.

In [5]:
examples=[
"I love Bangalore.",
"ನಾನು ಬೆಂಗಳೂರನ್ನು ಪ್ರೀತಿಸುತ್ತೇನೆ.",
"मुझे बेंगलुरु पसंद है।",
"எனக்கு பெங்களூரு பிடிக்கும்."
]

for s in examples:
    print(f"{len(enc.encode(s)):>3} tokens -> {s}")


  4 tokens -> I love Bangalore.
 59 tokens -> ನಾನು ಬೆಂಗಳೂರನ್ನು ಪ್ರೀತಿಸುತ್ತೇನೆ.
 25 tokens -> मुझे बेंगलुरु पसंद है।
 37 tokens -> எனக்கு பெங்களூரு பிடிக்கும்.


# 2️⃣ Estimate API Cost

Approximate pricing example only.
Always verify the latest pricing from the provider.


In [8]:
USD_TO_INR = 83       # Illustrative exchange rate
INPUT = 0.15          # Illustrative input price per 1M tokens
OUTPUT = 0.60         # Illustrative output price per 1M tokens

def estimate_cost(inp,out):
    usd=(inp/1_000_000)*INPUT+(out/1_000_000)*OUTPUT
    return round(usd,6),round(usd*USD_TO_INR,4)

usd, inr = estimate_cost(1000, 500)

print(f"Estimated cost in USD: ${usd}")
print(f"Estimated cost in INR: ₹{inr}")


Estimated cost in USD: $0.00045
Estimated cost in INR: ₹0.0374


# 3️⃣ Next Token Prediction

In [9]:
def ask(prompt,temperature=0.3,model="gpt-4o-mini",system="You are a helpful AI assistant."):
    response=client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role":"system","content":system},
            {"role":"user","content":prompt}
        ]
    )
    return response.choices[0].message.content

print(ask("Explain next-token prediction in three simple sentences."))


Next-token prediction is a task in natural language processing where a model predicts the next word or token in a sequence based on the preceding context. It involves analyzing patterns and relationships in the text to generate coherent and contextually relevant continuations. This technique is commonly used in applications like text generation, chatbots, and language translation.


# 4️⃣ Temperature

Lower = More deterministic

Higher = More creative


In [12]:
prompt = """
Suggest one memorable title for an Agentic AI faculty development programme.
Avoid generic phrases such as 'Introduction to AI'.
"""

for temp in [0.0,0.3,0.7,1.2]:
    print("="*60)
    print("Temperature:",temp)
    print(ask(prompt,temperature=temp))


Temperature: 0.0
"Empowering Minds: Cultivating Agentic AI in Education"
Temperature: 0.3
"Empowering Minds: Cultivating Agentic AI in Education"
Temperature: 0.7
"Empowering Minds: Navigating the Future with Agentic AI"
Temperature: 1.2
"Empowering Minds: Navigating the Future with Agentic AI"


# 5️⃣ Context Window

Context Window = The model's working memory.

Discussion:
- Why can't we keep sending unlimited documents?
- What are the trade-offs?


In [11]:
context_df = pd.DataFrame([
    {"Model family": "GPT-4o", "Illustrative context": "128K"},
    {"Model family": "Claude", "Illustrative context": "200K"},
    {"Model family": "Gemini", "Illustrative context": "1M+"}
])

context_df


,Model family,Illustrative context
0,GPT-4o,128K
1,Claude,200K
2,Gemini,1M+


# 6️⃣ Hallucination

A hallucination occurs when the model confidently generates incorrect information.


In [ ]:
q='Who won the Best AI Research Award at Sumeru Institute of Technology in 2026?'
print(ask(q,temperature=0.7))


### Safer System Prompt

In [ ]:
system='''
You are an academic assistant.
Never invent facts.
If information cannot be verified, clearly state that you do not know.
'''

print(ask(q,temperature=0.2,system=system))


# 7️⃣ AI Pulse – DeepSeek

Discussion Questions:
- What happened?
- Why did NVIDIA react?
- Does model size still matter?
- What does this mean for universities?


# 🎯 Lab Exercises

1. Tokenize your college name.
2. Compare English vs Indian language token counts.
3. Estimate API cost.
4. Experiment with temperature.
5. Trigger and reduce hallucination.
6. Discuss why RAG is needed.


# ✅ Key Takeaways

- LLMs predict the next token.
- Tokens drive cost and context.
- Temperature controls randomness.
- Hallucinations are a limitation.
- Prompting helps but RAG is required for grounding.
- Agents build on these foundations.
